In [65]:
import os
import h5py
import numpy as np
from scipy.stats import gaussian_kde
from scipy.interpolate import interp1d
from collections import defaultdict
snr = 50
# -------------------------
# PARAMETERS
# -------------------------
folder = "../proxima/results"
target_snr = "snr"+str(snr)  # look for filenames containing this string

# -------------------------
# LOAD DATA ORDER-WISE
# -------------------------
# Structure: combined_data[(snr, nspec)][order][seed] = {
#    "intrinsic": (mean, unc),
#    "template":  (mean, unc),
#    "prior":     (mean, unc),
#    "mala":      array_of_samples
# }
combined_data = defaultdict(lambda: defaultdict(dict))
all_data = {}  
for filename in os.listdir(folder):
    if target_snr not in filename:
        continue
    if filename[0:2]!='ph':
        continue

    # Extract order from filename, e.g., "o20"
    order = int(filename.split('_')[1][1:])
    nspec = int(filename.split('_')[-1][-5:-3])
    if nspec!=20:
        continue
    filepath = os.path.join(folder, filename)
    print(filepath)
    # with h5py.File(filepath, "r") as f:
    #     nspec = f['Order']['Observational Parameters'].attrs['nspec']

    # print(filepath)
    with h5py.File(filepath, "r") as f:
        params = f['Order']['Observational Parameters'].attrs
        snr = params['snr']
        nspec = params['nspec']
        # order = f['Order'].attrs['order']
        print(order)
        rv_group = f['Order']['Observational Parameters']['RV Samples']

# Read observational params
        i = f['Order']['Observational Parameters'].attrs['i']
        snr = f['Order']['Observational Parameters'].attrs['snr']
        nspec = f['Order']['Observational Parameters'].attrs['nspec']

        # Spectrum (optional, if you need it)
        spectrum_group = f['Order']['Observational Parameters']['Spectrum']
        spectrum_analysis = {name: dset[:] for name, dset in spectrum_group.items()}

        # RV Samples
        rv_group = f['Order']['Observational Parameters']['RV Samples']
        rv_analysis = {
            seed: {subname: subdset[:] for subname, subdset in seed_group.items()}
            for seed, seed_group in rv_group.items()
        }
        print(rv_analysis)
        # Store under (snr, nspec, step)
        all_data[(order, snr, nspec)] = {
            "rv": rv_analysis,
        }

        wgrid = f['Order']['wgrid'][:] 
        non_ones = f['Order']['non_ones'][:] 
        inst_wgrid = f['Order']['inst_wgrid'][:]


../proxima/results/phoenix_o14_i0_snr50_nspec20.h5
14
{'seed_0': {'phoenix_rv': array([-552.43325609, -763.615969  , -613.71504245, -699.67666668,
       -659.43014823, -651.11053942, -645.25094517, -742.85393934,
       -761.4528758 , -594.64704861, -806.82957604, -765.36195753,
       -576.77704633, -761.365184  , -689.04029698, -599.67606545,
       -633.20401072, -646.6834976 , -693.6052263 , -801.09275318,
       -580.83353923, -812.86760304, -741.04238797, -731.2773628 ,
       -589.57461388, -659.09782375, -496.69624363, -527.51232012,
       -657.73476959, -701.81751897, -642.57589955, -729.57649071,
       -664.30016533, -676.34267235, -863.10154967, -631.01646073,
       -694.87328832, -592.86374449, -733.56781563, -637.55039906,
       -703.28845655, -606.58119023, -590.79579085, -700.09896747,
       -631.02047584, -726.63847468, -746.92574168, -589.40631937,
       -624.68090971, -593.01037331, -414.92705763, -619.01270482,
       -620.99728489, -636.32109608, -562.4877098

In [66]:
all_data.keys()

dict_keys([(14, 50, 20), (15, 50, 20), (16, 50, 20), (17, 50, 20), (8, 50, 20), (9, 50, 20), (6, 50, 20), (7, 50, 20), (4, 50, 20), (5, 50, 20), (3, 50, 20), (2, 50, 20), (18, 50, 20), (19, 50, 20), (30, 50, 20), (29, 50, 20), (31, 50, 20), (28, 50, 20), (32, 50, 20), (33, 50, 20), (35, 50, 20), (34, 50, 20), (20, 50, 20), (21, 50, 20)])

In [67]:
import numpy as np
import matplotlib.pylab as plt
from scipy.stats import norm

def weighted_rv_average(data):
    print("here")
    orders = sorted(data.keys())
    seeds = list(data[orders[0]]['rv'].keys())
    print(seeds)
    
    combined = {'rv': {}}

    for seed in seeds:
        print("here")
        combined['rv'][seed] = {}
        
        # Intrinsic RV
        rv_arrays = np.array([data[o]['rv'][seed]['phoenix_rv'] for o in orders])
        unc_arrays = np.array([data[o]['rv'][seed]['phoenix_uncertainty'] for o in orders])
        inv_var = 1 / unc_arrays**2

        weighted_rv = np.sum(rv_arrays * inv_var, axis=0) / np.sum(inv_var, axis=0)
        combined_unc = 1 / np.sqrt(np.sum(inv_var, axis=0))
        print(rv_arrays.shape)
        combined['rv'][seed]['phoenix_rv'] = weighted_rv
        combined['rv'][seed]['phoenix_uncertainty'] = combined_unc
        
        # # Template RV (and uncertainty) — same method
        # rv_arrays = np.array([data[o]['rv'][seed]['template_rv'] for o in orders])
        # unc_arrays = np.array([data[o]['rv'][seed]['template_uncertainty'] for o in orders])
        # inv_var = 1 / unc_arrays**2

        # weighted_rv = np.sum(rv_arrays * inv_var, axis=0) / np.sum(inv_var, axis=0)
        # combined_unc = 1 / np.sqrt(np.sum(inv_var, axis=0))
        # combined['rv'][seed]['template_rv'] = weighted_rv
        # combined['rv'][seed]['template_uncertainty'] = combined_unc


        # # Mala
        # per_order_samples = [np.asarray(data[o]['rv'][seed]['mala_samples']) for o in orders]  
        # mrv_arrays = np.mean(per_order_samples,axis=(1,2) ) 
        # munc_arrays = np.std(per_order_samples,axis=(1,2) ) 
        # minv_var = 1 / munc_arrays**2
        # mweighted_rv = np.sum(mrv_arrays * minv_var, axis=0) / np.sum(minv_var, axis=0)
        # mcombined_unc = 1 / np.sqrt(np.sum(minv_var, axis=0)) 
        # pooled = np.concatenate(per_order_samples)
        # combined['rv'][seed]['mala_rv'] = mweighted_rv
        # combined['rv'][seed]['mala_uncertainty'] = mcombined_unc
        # # combined['rv'][seed]['mala_samples'] = pooled
        combined['rv'][seed]['true_planet'] = data[(20,snr,20)]['rv'][seed]['true_planet']
        # if seed=='seed_1':
        #     for i in range(8):
        #         obs = 20

        #         plt.hist(per_order_samples[i][:,:,obs],bins=25,alpha=0.75,label=orders[i],density=True)
            
        #     # plt.hist(pooled[:,:,obs],bins=25,alpha=0.75,color="k",label="MALA",density=True)
        #     x = np.linspace(-500, 500, 400)
        #     gaussian = norm.pdf(x, loc=mweighted_rv[obs], scale=mcombined_unc[obs])
        #     plt.plot(x,gaussian,label='Weighted Mala',c="k")
        #     plt.legend()
        #     plt.xlabel("RV (m/s)")
        #     plt.xlim(-200,200)
        #     plt.show()
        #     for i in range(8):
        #         x = np.linspace(-500, 500, 400)
        #         gaussian = norm.pdf(x, loc=rv_arrays[i][obs], scale=unc_arrays[i][obs])
        #         plt.plot(x,gaussian,label=orders[i][0])
        #     gaussian = norm.pdf(x, loc=weighted_rv[obs], scale=combined_unc[obs])
        #     plt.plot(x,gaussian,label='Weighted Template',c="k")
        #     plt.legend()
        #     plt.xlim(-200,200)
        #     plt.xlabel("RV (m/s)")
        #     plt.show()
        
    return combined
combined = weighted_rv_average(all_data)

here
['seed_0', 'seed_1', 'seed_2', 'seed_3', 'seed_4', 'seed_5', 'seed_6', 'seed_7', 'seed_8', 'seed_9']
here
(24, 100)
here
(24, 100)
here
(24, 100)
here
(24, 100)
here
(24, 100)
here
(24, 100)
here
(24, 100)
here
(24, 100)
here
(24, 100)
here
(24, 100)


In [68]:
import h5py
import numpy as np

def save_combined_to_h5(combined, i, snr, nspec, outfile):
    """
    Save the combined dictionary into an HDF5 file
    with the same structure as the original per-order files.
    """

    with h5py.File(outfile, "w") as f:

        # --- Main structure ---
        order_group = f.create_group("Order")
        obs_group = order_group.create_group("Observational Parameters")

        # --- Write observational parameters as attributes ---
        obs_group.attrs["i"] = i
        obs_group.attrs["snr"] = snr
        obs_group.attrs["nspec"] = nspec   # total number of combined RV points

        # --- RV Samples ---
        rv_group = obs_group.create_group("RV Samples")

        for seed in combined["rv"]:
            seed_group = rv_group.create_group(seed)

            # intrinsic rv + uncertainty
            seed_group.create_dataset("phoenix_rv", data=combined["rv"][seed]["phoenix_rv"])
            seed_group.create_dataset("phoenix_uncertainty", data=combined["rv"][seed]["phoenix_uncertainty"])

            # # template rv + uncertainty
            # seed_group.create_dataset("template_rv", data=combined["rv"][seed]["template_rv"])
            # seed_group.create_dataset("template_uncertainty", data=combined["rv"][seed]["template_uncertainty"])
            seed_group.create_dataset("true_planet", data=combined["rv"][seed]["true_planet"])
            # # --- MALA samples ---
            # # combined mala_samples has shape [N_values, N_samples]
            # # original format was [1, N_values, N_samples]
            # # mala_out = combined["rv"][seed]["mala_samples"][np.newaxis, :, :]
            # # seed_group.create_dataset("mala_samples", data=mala_out)
            # seed_group.create_dataset("mala_rv", data=combined["rv"][seed]["mala_rv"])
            # seed_group.create_dataset("mala_uncertainty", data=combined["rv"][seed]["mala_uncertainty"])

    print(f"Saved combined file to: {outfile}")
save_combined_to_h5(combined,0,snr,20,"phoenix_proxima_i0_snr"+str(snr)+"_nspec20.h5")

Saved combined file to: phoenix_proxima_i0_snr50_nspec20.h5
